In [0]:
from datetime import datetime
import uuid

def log_bronze(run_id, resource, load_date, status, start_time, count, message):

    end_time = datetime.now()

    data = [(run_id, resource, load_date, status, start_time, end_time, count, message)]

    df = spark.createDataFrame(
        data,
        ["run_id","resource","load_date","status","start_time","end_time","record_count","message"]
    )

    df.write.mode("append").saveAsTable("fhir_catalog.audit.bronze_log")

In [0]:
def get_available_dates(resource):

    base_path = f"/Volumes/fhir_catalog/raw/fhir_raw_vol/{resource.lower()}/json/"
    paths = dbutils.fs.ls(base_path)

    dates = []
    for p in paths:
        if "load_date=" in p.name:
            dates.append(p.name.split("=")[1].replace("/", ""))

    return sorted(dates)

In [0]:
def get_processed_dates(resource):

    df = spark.sql(f"""
        SELECT load_date
        FROM fhir_catalog.audit.bronze_log
        WHERE resource = '{resource}' AND status = 'SUCCESS'
    """)

    return [row[0] for row in df.collect()]

def get_pending_dates(resource):

    available = set(get_available_dates(resource))
    processed = set(get_processed_dates(resource))

    return sorted(list(available - processed))

In [0]:
from pyspark.sql.functions import from_json, col, explode, schema_of_json

def bronze_transform_common(resource):

    run_id = str(uuid.uuid4())

    pending_dates = get_pending_dates(resource)

    if not pending_dates:
        print(f"{resource}: No new data")
        return

    bronze_path = f"/Volumes/fhir_catalog/bronze/fhir_bronze_vol/{resource.lower()}/parquet/"

    for load_date in pending_dates:

        start_time = datetime.now()

        try:
            raw_path = f"/Volumes/fhir_catalog/raw/fhir_raw_vol/{resource.lower()}/json/load_date={load_date}/"

            df = spark.read.json(raw_path)

            if df.limit(1).count() == 0:
                log_bronze(run_id, resource, load_date, "SUCCESS", start_time, 0, "No data")
                continue

            sample_json = df.select("data").limit(1).collect()[0][0]
            schema = schema_of_json(sample_json)

            parsed_df = df.withColumn(
                "json_data",
                from_json(col("data"), schema)
            )

            flat_df = parsed_df.select(
                col("api_url"),
                col("extraction_timestamp"),
                explode("json_data.entry").alias("entry")
            )

            bronze_df = flat_df.select(
                col("entry.resource.id").alias("id"),
                col("entry.resource").alias("resource_data"),
                col("api_url"),
                col("extraction_timestamp"),
                col("extraction_timestamp").substr(1,10).alias("load_date")
            )

            count = bronze_df.count()

            bronze_df.write.mode("overwrite").parquet(bronze_path)

            log_bronze(run_id, resource, load_date, "SUCCESS", start_time, count, "Processed")

            print(f"{resource} {load_date} completed")

        except Exception as e:

            log_bronze(run_id, resource, load_date, "FAILED", start_time, 0, str(e))

            print(f"{resource} {load_date} FAILED")

In [0]:
resources = ["Patient", "Encounter", "Observation", "Condition"]

for r in resources:
    bronze_transform_common(r)